# Map of Italian Science — Country-Level Citation Analysis

**Research questions addressed in this notebook**

1. **(RQ1 – Map of Italian Science)** Which countries either cite or are cited by IRIS publications indexed in OpenCitations, for each of six Italian institutions?
2. **(RQ1a / RQ1b)** Do different institutions show different inbound and outbound citation patterns?

**Institutions covered:** UNIBO · UNIMI · UNIPD · UNITO · UPO · SNS

**Data:** For each institution, two CSV files are available:
- `citation_counts_countries_inbound.csv` — countries whose publications *cite* the institution's IRIS output
- `citation_counts_countries_outbound.csv` — countries *cited by* the institution's IRIS output

**Expected columns in each CSV:** `country_code` (ISO 2-letter), `country_name`, `count`

---
> **Notebook structure**
> 1. Setup & data loading
> 2. Single-institution deep-dive: UNIBO
>    - 2a. Inbound inspection
>    - 2b. Outbound inspection
>    - 2c. Diverging bar chart (inbound vs outbound)
>    - 2d. Choropleth maps
> 3. Cross-institution comparison (all 6 institutions)
>    - 3a. Top-N countries per institution
>    - 3b. Asymmetry analysis
>    - 3c. Shared vs unique citation partners
>    - 3d. Heatmap: institutions × countries
> 4. Summary of findings


## 1. Setup & Data Loading

In [111]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pycountry
from plotly.subplots import make_subplots
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH = Path("../data/citation_counts")  

INSTITUTIONS = ["UNIBO", "UNIMI", "UNIPD", "UNITO", "UPO", "SNS"]

INSTITUTION_LABELS = {
    "UNIBO": "University of Bologna",
    "UNIMI": "University of Milan",
    "UNIPD": "University of Padua",
    "UNITO": "University of Turin",
    "UPO":   "University of Eastern Piedmont",
    "SNS":   "Scuola Normale Superiore",
}

# ── Colour palette (one colour per institution, consistent across all plots) ──
INST_COLORS = {
    "UNIBO": "#264653",
    "UNIMI": "#2a9d8f",
    "UNIPD": "#8ab17d",
    "UNITO": "#e9c46a",
    "UPO":   "#f4a261",
    "SNS":   "#e76f51",
}

# ── Direction colours ─────────────────────────────────────────────────────────
DIR_COLORS = {"inbound": "#B7990D", "outbound": "#320E3B"}


### Data cleaning 

The following is a discovery cell to find all duplicates across all six institutions. The output is used to populate a manual mapping diciotnary, COUNTRY_NAMES.

In [ ]:
# Run once to find all country_code duplicates 
for inst in INSTITUTIONS:
    for direction in ["inbound", "outbound"]:
        raw = pd.read_csv(BASE_PATH / inst / f"citation_counts_countries_{direction}.csv")
        raw = raw.dropna(subset=["country_code"])
        dupes = raw.groupby("country_code")["country_name"].nunique()
        dupes = dupes[dupes > 1].index
        if len(dupes) > 0:
            print(f"\n{inst} {direction}:")
            print(raw[raw["country_code"].isin(dupes)]
                  [["country_code", "country_name"]]
                  .drop_duplicates()
                  .sort_values("country_code"))


UNIBO inbound:
    country_code                            country_name
214           CD          Congo (Democratic Republic of)
133           CD                                DR Congo
140           CI                             Ivory Coast
201           CI                           Cote d'Ivoire
28            CN            China (People's Republic of)
4             CN                                   China
205           CV                              Cabo Verde
237           CV                              Cape Verde
30            CZ                                 Czechia
87            CZ                          Czech Republic
94            IR              Iran (Islamic Republic of)
25            IR                                    Iran
177           KP                             North Korea
230           KP  Korea, Democratic People's Republic of
68            KR                     Korea (Republic of)
14            KR                             South Korea
173           L

In [ ]:
# Canonical country names for ambiguous cases (following common English usage)
COUNTRY_NAMES = {
    "BN": "Brunei",
    "CD": "DR Congo",
    "CG": "Congo Republic",
    "CI": "Ivory Coast",
    "CN": "China",
    "CV": "Cabo Verde",
    "CZ": "Czechia",
    "IR": "Iran",
    "KP": "North Korea",
    "KR": "South Korea",
    "LA": "Laos",
    "LY": "Libya",
    "MD": "Moldova",
    "MK": "North Macedonia",
    "NL": "The Netherlands",
    "PS": "Palestine",
    "RU": "Russia",
    "SY": "Syria",
    "SZ": "Eswatini",
    "TR": "Turkey",         
    "TZ": "Tanzania",
    "VI": "U.S. Virgin Islands",
    "VN": "Vietnam",
    "XK": "Kosovo",
}

In [112]:
# ── Generic loader ────────────────────────────────────────────────────────────
def load_country_data(institution: str, direction: str) -> pd.DataFrame:
    """Load a citation-counts CSV for one institution and direction.
    
    Parameters
    ----------
    institution : str  e.g. 'UNIBO'
    direction   : str  'inbound' or 'outbound'
    
    Returns
    -------
    pd.DataFrame with columns: country_code, country_name, count, institution, direction
    """
    path = BASE_PATH / institution / f"citation_counts_countries_{direction}.csv"
    df = pd.read_csv(path)

    # ── Cleaning ──────────────────────────────────────────────────────────────
    # Drop rows with missing country_code or country_name 
    df = df.dropna(subset=["country_code", "country_name"])
    df = df[df["country_code"].str.strip() != ""]

    # Normalize country_code formatting
    df["country_code"] = df["country_code"].str.strip().str.upper()

    # Apply canonical name mapping (only affects ambiguous cases)
    df["country_name"] = df["country_code"].map(COUNTRY_NAMES).fillna(df["country_name"])

    # Aggregate: sum counts for rows that now share the same (code, name) after mapping
    df = (df.groupby(["country_code", "country_name"], as_index=False)["count"]
            .sum())

    # ── Add metadata ──────────────────────────────────────────────────────────
    df["institution"] = institution
    df["direction"]   = direction

    return df

def load_institution(institution: str,
                     exclude_self: bool = True) -> pd.DataFrame:
    """Load and merge inbound + outbound for one institution.
    
    Parameters
    ----------
    exclude_self : bool
        If True, drop Italy (IT) to focus on *international* relationships.
    """
    inbound  = load_country_data(institution, "inbound")
    outbound = load_country_data(institution, "outbound")
    
    df = pd.concat([inbound, outbound], ignore_index=True)
    
    if exclude_self:
        df = df[df["country_code"] != "IT"]
    
    return df


def load_all(exclude_self: bool = True) -> pd.DataFrame:
    """Load data for ALL institutions into one long-format DataFrame."""
    frames = [load_institution(inst, exclude_self=exclude_self)
              for inst in INSTITUTIONS]
    return pd.concat(frames, ignore_index=True)


# ── Pivot helper ──────────────────────────────────────────────────────────────
def pivot_directions(df: pd.DataFrame) -> pd.DataFrame:
    """Pivot a long-format single-institution DataFrame to wide format.
    
    Returns a DataFrame with columns:
      country_code, country_name, inbound_count, outbound_count,
      total, difference (outbound − inbound), ratio (out/in)
    """
    wide = df.pivot_table(
        index=["country_code", "country_name"],
        columns="direction",
        values="count",
        aggfunc="sum"
    ).reset_index()
    wide.columns.name = None
    wide = wide.rename(columns={"inbound": "inbound_count",
                                 "outbound": "outbound_count"})
    wide = wide.fillna(0)
    wide["total"]      = wide["inbound_count"] + wide["outbound_count"]
    wide["difference"] = wide["outbound_count"] - wide["inbound_count"]
    # ratio: log-scale-friendly asymmetry: log2(out/in)
    # guard against zeros
    wide["log2_ratio"] = np.log2(
        (wide["outbound_count"] + 1) / (wide["inbound_count"] + 1)
    )
    return wide.sort_values("total", ascending=False)


# ── Converter for country codes ──────────────────────────────────────────────────────────────
def to_iso3(code2):
    """Convert ISO 3166-1 alpha-2 to alpha-3. Returns None if not found."""
    try:
        return pycountry.countries.get(alpha_2=code2).alpha_3
    except AttributeError:
        return None


In [113]:
# ── Verification: check cleaning worked correctly ─────────────────────────────

test = load_country_data("UNIBO", "inbound")

# 1. No duplicate country codes
dupes = test[test.duplicated(subset=["country_code"], keep=False)]
if dupes.empty:
    print("✓ No duplicate country codes")
else:
    print("✗ Duplicate country codes found:")
    print(dupes)

# 2. No missing values
if test[["country_code", "country_name", "count"]].isnull().any().any():
    print("✗ Missing values found")
else:
    print("✓ No missing values")

# 3. Spot-check known problematic codes resolve correctly
expected = {
    "RU": "Russia",
    "CN": "China",
    "TR": "Turkey",
    "NL": "The Netherlands",
    "CZ": "Czechia",
    "KR": "South Korea",
}
print("\nSpot-check canonical names:")
for code, expected_name in expected.items():
    row = test[test["country_code"] == code]
    if row.empty:
        print(f"  {code}: NOT FOUND in data")
    elif row.iloc[0]["country_name"] == expected_name:
        print(f"  ✓ {code} → {row.iloc[0]['country_name']}")
    else:
        print(f"  ✗ {code} → {row.iloc[0]['country_name']} (expected {expected_name})")

# 4. Check counts were summed correctly for a known duplicate
# Russia + Russian Federation should now be one row with combined count
print(f"\nRussia combined count: {test[test['country_code'] == 'RU']['count'].values[0]:,}")

# 5. Run the same check across ALL institutions and directions
print("\n── Full scan for remaining duplicates across all files ──")
found_issues = False
for inst in INSTITUTIONS:
    for direction in ["inbound", "outbound"]:
        df = load_country_data(inst, direction)
        dupes = df[df.duplicated(subset=["country_code"], keep=False)]
        if not dupes.empty:
            print(f"✗ {inst} {direction}: duplicates found")
            print(dupes)
            found_issues = True
if not found_issues:
    print("✓ No duplicates found in any institution or direction")

✓ No duplicate country codes
✓ No missing values

Spot-check canonical names:
  ✓ RU → Russia
  ✓ CN → China
  ✓ TR → Turkey
  ✓ NL → The Netherlands
  ✓ CZ → Czechia
  ✓ KR → South Korea

Russia combined count: 612,433

── Full scan for remaining duplicates across all files ──
✓ No duplicates found in any institution or direction


In [114]:
# ── Load everything ───────────────────────────────────────────────────────────
# Long-format, Italy excluded, all institutions
all_df = load_all(exclude_self=True)

print(f"Total rows: {len(all_df):,}")
print(f"Institutions: {all_df['institution'].unique()}")
print(f"Directions:   {all_df['direction'].unique()}")
all_df.head()


Total rows: 2,591
Institutions: ['UNIBO' 'UNIMI' 'UNIPD' 'UNITO' 'UPO' 'SNS']
Directions:   ['inbound' 'outbound']


,country_code,country_name,count,institution,direction
0,AD,Andorra,19,UNIBO,inbound
1,AE,United Arab Emirates,19714,UNIBO,inbound
2,AF,Afghanistan,432,UNIBO,inbound
3,AG,Antigua and Barbuda,1952,UNIBO,inbound
4,AL,Albania,1403,UNIBO,inbound


## 2. Single-Institution Deep-Dive: UNIBO

Before comparing all six institutions we examine UNIBO in detail to:
- understand the data structure and scale
- surface methodological decisions (log scaling, Italy exclusion)
- produce candidate visualizations

### 2a. Inbound — Who cites UNIBO?


In [115]:
unibo_df = load_institution("UNIBO", exclude_self=True)
unibo_wide = pivot_directions(unibo_df)

unibo_wide.head(10)

,country_code,country_name,inbound_count,outbound_count,total,difference,log2_ratio
212,US,United States,6800261.0,8894677.0,15694938.0,2094416.0,0.387352
68,FR,France,3956305.0,4043901.0,8000206.0,87596.0,0.031594
70,GB,United Kingdom,1986678.0,2461117.0,4447795.0,474439.0,0.308955
50,DE,Germany,1801165.0,1828283.0,3629448.0,27118.0,0.021559
42,CN,China,2167251.0,878920.0,3046171.0,-1288331.0,-1.302062
60,ES,Spain,1262588.0,1051307.0,2313895.0,-211281.0,-0.264200
102,JP,Japan,889572.0,842595.0,1732167.0,-46977.0,-0.078272
151,NL,The Netherlands,679657.0,800205.0,1479862.0,120548.0,0.235562
34,CA,Canada,676387.0,761331.0,1437718.0,84944.0,0.170675
11,AU,Australia,661316.0,670465.0,1331781.0,9149.0,0.019822


In [116]:
unibo_df = load_institution("UNIBO", exclude_self=True)
unibo_wide = pivot_directions(unibo_df)

# Inbound top-15
inbound_top15 = (unibo_df[unibo_df["direction"] == "inbound"]
                 .sort_values("count", ascending=False)
                 .head(15))

fig = px.bar(
    inbound_top15,
    x="count", y="country_name",
    orientation="h",
    title="Top 15 Countries Citing UNIBO (inbound, excl. Italy)",
    labels={"count": "Citation count", "country_name": ""},
    color_discrete_sequence=[DIR_COLORS["inbound"]],
    template="plotly_white",
    height=520
)
fig.update_layout(title_x=0.5)
fig.show()


**Observations — UNIBO inbound**

- The distribution is **highly right-skewed**: the United States alone contributes millions of citations, far ahead of all other countries.
- France and China follow, together with United Kingdom and Germany: this suggests that UNIBO’s publications are primarily cited by countries with very large research ecosystems, high publication output and strong integration in international science. This was expected, but still owrth mentioning.
- Excluding Italy is justified here: Italy's self-citations would dominate and obscure other international patterns.
- The long tail (many countries with tiny counts) has important implications for visualisation: a linear colour scale on a choropleth will flatten everything outside the US. Logarithmic scaling is strongly recommended (see §2d).


### 2b. Outbound — Whom does UNIBO cite?

In [ ]:
outbound_top15 = (unibo_df[unibo_df["direction"] == "outbound"]
                  .sort_values("count", ascending=False)
                  .head(15))

fig = px.bar(
    outbound_top15,
    x="count", y="country_name",
    orientation="h",
    title="Top 15 Countries Cited by UNIBO (outbound, excl. Italy)",
    labels={"count": "Citation count", "country_name": ""},
    color_discrete_sequence=[DIR_COLORS["outbound"]],
    template="plotly_white",
    height=520
)
fig.update_layout(title_x=0.5)
fig.show()


**Observations — UNIBO outbound**

- The United States dominates outbound even more strongly than inbound, suggesting an **epistemic dependence** on US-centered science production.
- France is roughly symmetric in both directions — a sign of **reciprocal exchange**.
- China shows a notable asymmetry: it appears strongly in inbound (Chinese researchers cite UNIBO) but much less in outbound. Possible interpretations:
  - Western-centric citation practices
  - Language/publication ecosystem differences
  - Disciplinary composition of UNIBO's output


### 2c. Diverging Bar Chart — Inbound vs Outbound

In [ ]:
top15_wide = unibo_wide.head(15).copy()

# Define the order: ascending 
country_order = (top15_wide.sort_values("total", ascending=True)["country_name"].tolist())

# Build long format for diverging chart
inbound_long = top15_wide[["country_name", "inbound_count"]].copy()
inbound_long["direction"] = "inbound"
inbound_long["value"]     = -inbound_long["inbound_count"]
inbound_long["count"]     = inbound_long["inbound_count"]

outbound_long = top15_wide[["country_name", "outbound_count"]].copy()
outbound_long["direction"] = "outbound"
outbound_long["value"]     = outbound_long["outbound_count"]
outbound_long["count"]     = outbound_long["outbound_count"]

diverging_df = pd.concat([
    inbound_long[["country_name", "direction", "value", "count"]],
    outbound_long[["country_name", "direction", "value", "count"]]
])

max_val = diverging_df["value"].abs().max()

fig = px.bar(
    diverging_df,
    x="value", y="country_name",
    color="direction",
    orientation="h",
    custom_data=["count", "direction"],
    color_discrete_map=DIR_COLORS,
    title="Inbound vs Outbound Citation Relationships — UNIBO (Top 15)",
    labels={"value": "← Inbound  |  Outbound →", "country_name": ""},
    template="plotly_white",
    height=560,
    category_orders={"country_name": country_order} 
)
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Citations: %{customdata[0]:,.0f}<br>Direction: %{customdata[1]}<extra></extra>"
)
fig.update_xaxes(range=[-max_val * 1.05, max_val * 1.05])
fig.update_layout(title_x=0.5, bargap=0.15, legend_title_text="")
fig.add_vline(x=0, line_width=1.5, line_color="gray")
fig.show()


**Interpretation**

The diverging chart makes **asymmetries immediately visible**. Countries where the two bars are roughly equal length (e.g. France, Germany) represent balanced bilateral exchange. Countries where outbound >> inbound (e.g. United States) reveal a **citation dependency**. Countries where inbound >> outbound (e.g. China) reveal an **asymmetric incoming influence**.


### 2d. Choropleth Maps — Geographic view (log scale)

In [119]:
# Merge with all countries (including zero-citation ones = NaN on map = grey)
# We work directly with the wide table

unibo_inbound_full  = (unibo_df[unibo_df["direction"] == "inbound"]
                       .copy())
unibo_outbound_full = (unibo_df[unibo_df["direction"] == "outbound"]
                       .copy())

# Add log-scaled count for better colour range
for df in [unibo_inbound_full, unibo_outbound_full]:
    df["log_count"]    = np.log10(df["count"].clip(lower=1))
    df["country_iso3"] = df["country_code"].apply(to_iso3)

# Drop rows where conversion failed (e.g. XK for Kosovo, which has no ISO-3)
unibo_inbound_full  = unibo_inbound_full.dropna(subset=["country_iso3"])
unibo_outbound_full = unibo_outbound_full.dropna(subset=["country_iso3"])

fig_in = px.choropleth(
    unibo_inbound_full,
    locations="country_iso3",
    locationmode="ISO-3", 
    color="log_count",
    hover_name="country_name",
    hover_data={"count": ":,", "log_count": False},
    color_continuous_scale="YlOrBr",
    title="Inbound Citations to UNIBO (log₁₀ scale, excl. Italy)",
    labels={"log_count": "log₁₀(citations)"},
    template="plotly_white",
    height=420
)
fig_in.update_layout(title_x=0.5)
fig_in.show()

fig_out = px.choropleth(
    unibo_outbound_full,
    locations="country_iso3",
    locationmode="ISO-3",
    color="log_count",
    hover_name="country_name",
    hover_data={"count": ":,", "log_count": False},
    color_continuous_scale="Purples",
    title="Outbound Citations from UNIBO (log₁₀ scale, excl. Italy)",
    labels={"log_count": "log₁₀(citations)"},
    template="plotly_white",
    height=420
)
fig_out.update_layout(title_x=0.5)
fig_out.show()


**Note on log scaling:** The raw counts span several orders of magnitude (the US has tens of millions; many countries have fewer than 100). A linear colour scale makes the map monochromatic. `log₁₀` compresses the range and reveals the geographic texture. Hover on any country to see the real count.

This choropleth maps do not add anything more to previous visualizations beside geographic/spatial dimension, however it does add visual impact, making the global dominance of the US, Western Europe and a few Asian countries immediately legible at a glance. 

### 2e. Asymmetry Map — Which countries does UNIBO cite more than they cite it back?

This map encodes the directionality of UNIBO's citation relationships rather than their volume. For each country, the metric is log₂(outbound/inbound): positive values (blue) mean UNIBO cites that country more than it is cited back; negative values (red) mean the country cites UNIBO more than UNIBO cites it. A value of +1 means UNIBO cites twice as much as it receives; −1 means it receives twice as much as it sends. The color scale is capped at ±3 (an 8-fold difference in either direction) to prevent extreme values from small-count countries from dominating the palette.

This map should be read alongside the diverging bar charts, not as a substitute. The bar charts show which countries matter most by volume; this map shows the directional balance across the entire world, including countries too small to appear in any top-N ranking.

In [131]:
# log2_ratio > 0 → UNIBO cites them more (outbound bias)
# log2_ratio < 0 → they cite UNIBO more (inbound bias)

unibo_wide["country_iso3"] = unibo_wide["country_code"].apply(to_iso3)

# Drop rows where conversion failed (e.g. XK for Kosovo, which has no ISO-3)
unibo_wide = unibo_wide.dropna(subset=["country_iso3"])

fig_asym = px.choropleth(
    unibo_wide,
    locations="country_iso3",
    locationmode="ISO-3",
    color="log2_ratio",
    hover_name="country_name",
    hover_data={"inbound_count": ":,",
                "outbound_count": ":,",
                "log2_ratio": ":.2f"},
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    range_color=[-3, 3],
    title="Citation Asymmetry — UNIBO (log₂ outbound/inbound)",
    labels={"log2_ratio": "log₂(out/in)"},
    template="plotly_white",
    height=440
)
fig_asym.update_layout(title_x=0.5)
fig_asym.show()


## Findings
The map is almost entirely **red**. The near-universal inbound dominance across Africa, Latin America, the Middle East, and most of Asia is the most striking finding. For the majority of countries in the world, UNIBO receives more citations than it generates — the opposite of what the bar charts might suggest if read in isolation.

The **United States** is the primary **exception**. The US is the only major country appearing in blue, confirming what the diverging bar charts showed: UNIBO's outbound citations to American science are so large that they outpace even the substantial inbound flow from the US. This reflects the structural weight of US-based journals and publication venues in the global citation economy.

**Northern Europe** is roughly balanced. Several Northern and Western European countries — visible as pale pink or near-white — sit close to zero, indicating reciprocal exchange. This is consistent with shared European research infrastructure and co-authorship networks.

The deep red in parts of **Africa** and **Asia** reflects low absolute volumes. Many of these countries have very few total citations in either direction, so even a small imbalance produces an extreme log ratio. These values should be interpreted cautiously: a country with 10 inbound citations and 1 outbound will appear as deep red (log₂ = −3.3) but the relationship is statistically negligible. The asymmetry map is most reliable for countries that also appear in the top-N bar charts, where volumes are large enough to make the ratio meaningful.

#### Overall interpretation. 
The asymmetry map reframes the narrative established by the bar charts. The outbound dominance toward the US and a handful of Western European countries is real but limited to a small number of high-volume partners. Globally, UNIBO — and by extension Italian science more broadly, as this pattern is consistent across institutions — is a net citation receiver. This is consistent with a peripheral position in the global citation network relative to Anglophone science production, while simultaneously being a significant cited source for researchers in the Global South and Asia.


---
## 3. Cross-Institution Comparison

Now we scale the same analyses to all six institutions to answer **RQ1a/RQ1b**: do institutions show different citation patterns?



### 3a. Proportional stacked bar chart

This is the starting point for investigating the international connections of the six Italian institutions. First, we establish the baseline geographic distribution of their citation networks. By calculating the proportional contribution of the top 10 cited and citing countries, we map the fundamental orientation of these institutions within the global scientific landscape.

This step is essential to verify whether the institutions operate within a shared international framework or if there are immediate, macro-level discrepancies in their reach.

In [127]:

def plot_proportional_citations(direction, title):
    all_data = []
    
    for inst in INSTITUTIONS:
        df = load_country_data(inst, direction)
        df = df[df["country_code"] != "IT"]
        all_data.append(df)   
        
    combined_df = pd.concat(all_data, ignore_index=True)
    
    top_10_countries = combined_df.groupby('country_name')['count'].sum().nlargest(10).index

    # Filter the dataframe to keep only those 10 countries
    final_df = combined_df[combined_df['country_name'].isin(top_10_countries)].copy()
    totals = final_df.groupby('institution')['count'].transform('sum')
    final_df['percentage'] = (final_df['count'] / totals) * 100

    colors_10 = [
        '#320E3B',
        '#4D1343',
        '#69184B',
        '#861D53',
        '#A3225B',
        '#BF2862',
        '#C84457',
        '#D0604D',
        '#C47D30',
        '#B7990D'
    ]

    # Create the Plotly chart
    fig = px.bar(
        final_df,
        x="institution",
        y="percentage",
        color="country_name",
        title=title,
        category_orders={"country_name": list(top_10_countries)}, 
        color_discrete_sequence=colors_10,
        labels={
            "Percentage": "Proportion among Top 10 (%)",
            "country_name": "Country",
            "Institution": "Institution"
        },
        hover_data={"count": True, "percentage": ':.2f'} 
    )

    fig.update_layout(
        barmode='stack',
        template="plotly_white",
        title_font=dict(size=18, family="Arial, sans-serif"),
        hoverlabel=dict(bgcolor="white", font_size=13),
        legend=dict(
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        )
    )

    fig.show()

# Citing entities (inbound)
plot_proportional_citations(
    direction='inbound', 
    title='International Geographic Distribution of Citing Entities (Top 10)'
)

# Cited entities (outbound)
plot_proportional_citations(
    direction='outbound', 
    title='International Geographic Distribution of Cited Entities (Top 10)'
)

### Results

At a macro level, the geographic distribution of both inbound (cited) and outbound (citing) citations reveals a highly isomorphic pattern across the six Italian universities. The stacked proportional bar charts confirm that these institutions are firmly integrated into the traditional global scientific core.

Specifically, the US, France and the UK consistently emerge as the dominant hubs, collectively accounting for the vast majority of international citation flows.

However, while these charts demonstrate a shared macro-level integration, they also mask the specific insitutional behavior hidden beneath these uniform volume aggregates. To further analyze the data and deepen our exploration we use alternatives highlighting how each insitution differs from the global network and other hidden patterns.


### 3b. Inbound vs. Outbound Citation Partners — Diverging Bar Charts

For each of the six institutions, the chart below displays the top 12 countries by total citation volume (inbound + outbound combined, Italy excluded), arranged as a diverging horizontal bar chart. Gold bars extend leftward and represent **inbound** citations — publications from that country that cite the institution's IRIS output indexed in OpenCitations. Dark purple bars extend rightward and represent **outbound** citations — references made by the institution's publications to works from that country.

Countries are sorted by total volume (most cited partners at the top). Each subplot has **an independent x-axis**, scaled symmetrically around zero to the maximum value observed for that institution. This choice makes asymmetries readable regardless of institutional size: the University of Bologna and University of Milan operate on a ±5M scale, while UPO and SNS operate on a ±1–2M scale.


In [ ]:
TOP_N = 12  # per institution; slightly more than before since both directions share the same bars

from plotly.subplots import make_subplots
import plotly.graph_objects as go

INST_ORDER = ["UNIBO", "UNIMI", "UNIPD", "UNITO", "UPO", "SNS"]
NROWS, NCOLS = 2, 3

fig = make_subplots(
    rows=NROWS, cols=NCOLS,
    subplot_titles=[INSTITUTION_LABELS[i] for i in INST_ORDER],
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

for idx, inst in enumerate(INST_ORDER):
    row = idx // NCOLS + 1
    col = idx % NCOLS + 1

    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top = wide.head(TOP_N).copy()

    # Sort ascending so the longest bar is at the top in the plot
    top = top.sort_values("total", ascending=True)

    color = INST_COLORS[inst]

    # Inbound bars go LEFT (negative x)
    fig.add_trace(go.Bar(
        x=-top["inbound_count"],
        y=top["country_name"],
        orientation="h",
        name="Inbound",
        marker_color=DIR_COLORS["inbound"],
        showlegend=(idx == 0),
        legendgroup="inbound",
        hovertemplate="<b>%{y}</b><br>Inbound: %{customdata:,}<extra></extra>",
        customdata=top["inbound_count"],
    ), row=row, col=col)

    # Outbound bars go RIGHT (positive x)
    fig.add_trace(go.Bar(
        x=top["outbound_count"],
        y=top["country_name"],
        orientation="h",
        name="Outbound",
        marker_color=DIR_COLORS["outbound"],
        showlegend=(idx == 0),
        legendgroup="outbound",
        hovertemplate="<b>%{y}</b><br>Outbound: %{customdata:,}<extra></extra>",
        customdata=top["outbound_count"],
    ), row=row, col=col)

    # Zero line per subplot
    fig.add_vline(x=0, line_width=1, line_color="gray", row=row, col=col)

fig.update_layout(
    title=f"Inbound vs Outbound Citation Partners — Top {TOP_N} Countries per Institution (excl. Italy)",
    title_x=0.5,
    barmode="overlay",
    template="plotly_white",
    height=850,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
        title_text=""
    )
)

# Symmerise x-axes so zero is centred — compute per institution
for idx, inst in enumerate(INST_ORDER):
    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top = wide.head(TOP_N)
    max_val = max(top["inbound_count"].max(), top["outbound_count"].max()) * 1.1
    xaxis_key = "xaxis" if idx == 0 else f"xaxis{idx+1}"
    fig.layout[xaxis_key].update(range=[-max_val, max_val])

fig.show()

### Findings

**Shared geographic structure.** The citation geography is strikingly consistent across all six institutions. The United States, France, the United Kingdom, Germany, China, and Spain appear in every institution's top 12 in both directions, suggesting that these partnerships reflect the broader structure of international science rather than institution-specific strategies.

**Systematic outbound dominance.** Across all institutions, outbound bars are consistently longer than inbound bars — particularly for the United States. Italian universities collectively cite American science far more than American science cites them back. This pattern is well-documented in the scientometrics literature and reflects both the volume of US publication output and citation practices within disciplines where US journals dominate.

**China asymmetry.** China is a notable exception to the outbound-dominant pattern. For most institutions, the inbound bar for China is visibly longer than or comparable to the outbound bar, meaning Chinese publications cite Italian research more than Italian publications cite Chinese research. This likely reflects a combination of Western-centric citation practices and the disciplinary composition of each institution's output.

**France as the most balanced partner.** France consistently shows the most symmetric bars across institutions — a sign of genuine bilateral exchange rather than a directional dependency. This likely reflects shared European research infrastructures and co-authorship networks.

**Institution-specific partners.** Below the top five countries, some divergence appears. **Russia** appears in the top 12 for SNS, UPO and UNIBO but not for the other three universities. Same for **India** which enters the top 12 for UPO, UNITO, SNS, and UNIPD but not for UNIBO or UNIMI. These divergences across otherwise similar large institutions is worth noting and may reflect disciplinary composition differences. 
**South Korea** and **Turkey** appear for SNS and UPO. These differences likely reflect disciplinary specialisations: SNS's concentration in mathematics and physics, and UPO's smaller size making niche international partnerships more visible in relative terms.

**Scale differences are meaningful.** The raw volume difference between UNIBO/UNIMI/UNIPD/UNITO (±5M range) and UPO/SNS (±1–2M range) reflects differences in institutional size and publication output rather than citation behaviour per se. Comparisons of *shape* (which countries appear, how symmetric the bars are) are more informative across institutions than comparisons of raw counts.

## 3c. Heatmap

As stated before, the stacked bar charts confirm broad integration into global networks, but they mask the institution-specific specializations that differentiate the research profiles of the six Italian institutions.


To highlight these nuances, we pivot to a relative specialization analysis. By calculating the deviation of each institution from the six-university mean, we can strip away the global noise created by the dominance of major countries like the US, and instead highlight the unique geographic fingerprints of each university.

In [130]:
def plot_heatmap(direction, title, top_n=15):
    all_data = []

    for inst in INSTITUTIONS:
        # Use the shared loader with cleaning pipeline
        df = load_country_data(inst, direction)
        df = df[df["country_code"] != "IT"]
        all_data.append(df)

    combined_df = pd.concat(all_data, ignore_index=True)

    # Calculate true proportions before filtering countries
    totals = combined_df.groupby("institution")["count"].transform("sum")
    combined_df["Proportion"] = (combined_df["count"] / totals) * 100

    # Identify Top N countries overall
    top_countries = (combined_df.groupby("country_name")["count"]
                     .sum()
                     .nlargest(top_n)
                     .index)

    # Filter data and pivot
    heatmap_df = combined_df[combined_df["country_name"].isin(top_countries)].copy()
    pivot_df = (heatmap_df.pivot(index="country_name", columns="institution", values="Proportion")
                .fillna(0))
    pivot_df = pivot_df[INSTITUTIONS]

    # Calculate Average and Deviation
    pivot_df["Average"] = pivot_df.mean(axis=1)
    deviation_df = pivot_df[INSTITUTIONS].sub(pivot_df["Average"], axis=0)

    deviation_df["Average"] = pivot_df["Average"]
    deviation_df = deviation_df.sort_values(by="Average", ascending=True)

    averages = deviation_df["Average"].values
    countries = deviation_df.index.tolist()

    y_labels_with_avg = [f"{country} (Avg: {avg:.1f}%)"
                         for country, avg in zip(countries, averages)]

    true_proportions = deviation_df[INSTITUTIONS].values + averages[:, None]

    deviation_df = deviation_df.drop(columns=["Average"])

    custom_colorscale = [
        [0.0, "#B7990D"],  # Under-indexing
        [0.5, "#FFFFFF"],  # Average
        [1.0, "#320E3B"]   # Over-indexing
    ]

    vmax = np.abs(deviation_df.values).max()

    fig = go.Figure(data=go.Heatmap(
        z=deviation_df.values,
        x=deviation_df.columns,
        y=y_labels_with_avg,
        customdata=true_proportions,
        colorscale=custom_colorscale,
        zmin=-vmax,
        zmax=vmax,
        zmid=0,
        hovertemplate=(
            "<b>Institution:</b> %{x}<br>" +
            "<b>Country:</b> %{y}<br>" +
            "<b>Deviation from Avg:</b> %{z:+.1f} pp<br>" +
            "<b>True Proportion:</b> %{customdata:.1f}%<br>" +
            "<extra></extra>"
        ),
        colorbar=dict(title="Deviation (pp)", titleside="right"),
        xgap=1,
        ygap=1
    ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family="Arial, sans-serif")),
        template="plotly_white",
        xaxis=dict(title="", tickfont=dict(size=12, weight="bold"), side="top"),
        yaxis=dict(title="", tickfont=dict(size=12)),
        width=850,
        height=600,
        margin=dict(t=100, l=180)
    )

    annotations = []
    for i, row in enumerate(deviation_df.values):
        for j, val in enumerate(row):
            text_color = "white" if abs(val) > (vmax * 0.6) else "black"
            annotations.append(dict(
                x=deviation_df.columns[j],
                y=y_labels_with_avg[i],
                text=f"{val:+.1f}",
                font=dict(color=text_color, size=10),
                showarrow=False
            ))

    fig.update_layout(annotations=annotations)
    fig.show()


# Inbound
plot_heatmap(
    direction="inbound",
    title="Relative Geographic Specialization: Inbound Citations (Top 15 Countries)"
)

# Outbound
plot_heatmap(
    direction="outbound",
    title="Relative Geographic Specialization: Outbound Citations (Top 15 Countries)"
)

### Results

By calculating each university's deviation from the group meanm, the relative specialization heatmaps reveal distinct institutional behaviors that are invisible in raw volume counts.

For instance, the Scuola Normale Superiore (SNS) shows a significantly higher reliance on the US - since the resulting deviation is +2.9pp from the average. Conversely, institutions such as UPO and UNITO exhibit a deeper structural alignment with Chinese research networks.

These deviations indicate that while the Italian science core operates within a unified global hierarchy, individual geographic spheres of influence might be dictated by other aspects, such as localized strategic priorities and specific disciplinary focuses.

### 3d. Asymmetry Comparison Across Institutions

In [ ]:
# For each institution compute the mean log2_ratio across its top countries
# as a summary measure of net citation balance

summary_rows = []
for inst in INSTITUTIONS:
    df = load_institution(inst, exclude_self=True)
    wide = pivot_directions(df)
    top20 = wide.head(20)
    summary_rows.append({
        "institution": inst,
        "label": INSTITUTION_LABELS[inst],
        "mean_log2_ratio": top20["log2_ratio"].mean(),
        "n_inbound_countries": (df[df["direction"]=="inbound"]["count"] > 0).sum(),
        "n_outbound_countries": (df[df["direction"]=="outbound"]["count"] > 0).sum(),
        "total_inbound": df[df["direction"]=="inbound"]["count"].sum(),
        "total_outbound": df[df["direction"]=="outbound"]["count"].sum(),
    })

summary = pd.DataFrame(summary_rows)
summary["total"] = summary["total_inbound"] + summary["total_outbound"]
summary["pct_outbound"] = summary["total_outbound"] / summary["total"] * 100
summary.set_index("institution", inplace=True)
summary


,label,mean_log2_ratio,n_inbound_countries,n_outbound_countries,total_inbound,total_outbound,total,pct_outbound
institution,,,,,,,,
UNIBO,University of Bologna,-0.076902,226,225,29872589,30124137,59996726,50.209635
UNIMI,University of Milan,-0.402483,227,222,32727946,27257280,59985226,45.439989
UNIPD,University of Padua,-0.165805,228,223,28571838,26963079,55534917,48.551579
UNITO,University of Turin,-0.030687,225,219,18823947,19447727,38271674,50.814937
UPO,University of Eastern Piedmont,0.032534,213,208,6401677,6717834,13119511,51.204912
SNS,Scuola Normale Superiore,0.006932,194,181,6455250,6531698,12986948,50.294326


In [ ]:
fig = px.bar(
    summary.reset_index(),
    x="institution",
    y="mean_log2_ratio",
    color="institution",
    color_discrete_map=INST_COLORS,
    title="Mean Citation Asymmetry per Institution (log₂ outbound/inbound, top 20 countries)",
    labels={"mean_log2_ratio": "Mean log₂(outbound/inbound)",
            "institution": ""},
    template="plotly_white",
    text_auto=".2f"
)
fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray",
              annotation_text="balanced", annotation_position="right")
fig.update_layout(title_x=0.5, showlegend=False, height=420)
fig.show()


**Interpretation:** A positive bar means the institution cites the rest of the world more than it is cited back (outbound-biased). A negative bar means it attracts more foreign citations than it generates (inbound-biased). Comparing this metric across institutions tests **RQ1a/RQ1b** directly.


### 3e. Volume vs. Diversity Scatter

In [ ]:
# A scatter where:
#   x = total citation volume (log scale)
#   y = number of distinct countries with at least 1 citation
#   shape = direction

scatter_rows = []
for inst in INSTITUTIONS:
    df = load_institution(inst, exclude_self=True)
    for direction in ["inbound", "outbound"]:
        sub = df[df["direction"] == direction]
        scatter_rows.append({
            "institution": inst,
            "label": INSTITUTION_LABELS[inst],
            "direction": direction,
            "total": sub["count"].sum(),
            "n_countries": (sub["count"] > 0).sum(),
        })

scatter_df = pd.DataFrame(scatter_rows)

fig = px.scatter(
    scatter_df,
    x="total", y="n_countries",
    color="institution",
    symbol="direction",
    text="institution",
    color_discrete_map=INST_COLORS,
    log_x=True,
    title="Citation Volume vs. Geographic Diversity by Institution & Direction",
    labels={"total": "Total citations (log scale)",
            "n_countries": "Number of distinct countries"},
    template="plotly_white",
    height=480
)
fig.update_traces(textposition="top center", marker_size=11)
fig.update_layout(title_x=0.5)
fig.show()


**Reading this chart:** Each institution appears twice (circle = inbound, triangle = outbound). Institutions in the top-right corner have both high volume and broad geographic spread. Institutions in the bottom-left are small and geographically concentrated. Divergence between the two symbols for the same institution indicates that one direction is more globally distributed than the other.


---
## 4. Summary of Findings

> **Fill this section in after running the cells above.** The placeholders below are prompts for what to look for.

### 4.1 Common patterns across all institutions
- [ ] Which countries appear in the top-10 of all 6 institutions in both directions? (Expected: US, UK, Germany, France)
- [ ] Is the US dominance uniform in scale or does it vary by institution?

### 4.2 Institution-specific patterns (RQ1a — inbound / RQ1b — outbound)
- [ ] Does SNS (Scuola Normale Superiore) show a more European-focused pattern, given its humanities/social science profile?
- [ ] Does UPO (a smaller regional university) show a more concentrated and less globally distributed pattern?
- [ ] Do the large research-intensive universities (UNIBO, UNIMI, UNIPD, UNITO) converge on similar patterns?

### 4.3 Asymmetry findings
- [ ] Which institutions are net citation exporters vs importers?
- [ ] Is the China asymmetry (cites Italian science but is less cited back) consistent across all institutions?

### 4.4 Methodological notes
- **Italy excluded** from all analyses to isolate international relationships.
- **Log₁₀ scaling** used on choropleth maps; linear axes retained on ranked bar charts for readability.
- **Log₂ ratio** used as asymmetry metric: it is symmetric around 0, robust to volume differences, and interpretable (log₂ = 1 means twice as many citations in one direction).
- Small institutions (UPO, SNS) will have lower total counts; comparisons of *patterns* (ranks, ratios) are more meaningful than comparisons of raw volume.
